# Визуализация данных

© Гошин Е.В., доцент технической кибернетики, Самарский университет  
© Петров М.В., старший преподаватель кафедры киберфотоники, Самарский университет

# Лекция 2. Преобразование данных в аккуратную (tidy) форму

## Содержание

1. [Введение](#2.1-Введение)
2. [Преобразование значений переменных, выступающих именами столбцов, с помощью `stack`](#22-преобразование-значений-переменных-выступающих-именами-столбцов-с-помощью-stack)
3. [Преобразование значений переменных, выступающих именами столбцов, с помощью `melt`](#23-преобразование-значений-переменных-выступающих-именами-столбцов-с-помощью-melt)
4. [Одновременное использование `stack` для нескольких групп переменных](#24-одновременное-использование-stack-для-нескольких-групп-переменных)
5. [Инвертирование данных, полученных `stack`/`melt`](#25-инвертирование-данных-полученных-stackmelt)
6. [Использование `unstack` после агрегации `groupby`](#26-использование-unstack-после-агрегации-groupby)
7. [Повторение функционала `pivot_table` с помощью агрегации `groupby`](#27-повторение-функционала-pivot_table-с-помощью-агрегации-groupby)
8. [Переименование уровней осей для удобного изменения формы данных](#28-переименование-уровней-осей-для-удобного-изменения-формы-данных)
9. [Приведение к аккуратной форме, когда несколько переменных хранятся в именах столбцов](#29-приведение-к-аккуратной-форме-когда-несколько-переменных-хранятся-в-именах-столбцов)
10. [Приведение к аккуратной форме, когда несколько переменных хранятся в одном столбце](#210-приведение-к-аккуратной-форме-когда-несколько-переменных-хранятся-в-одном-столбце)
11. [Приведение к аккуратной форме, когда в одной ячейке хранится два и более значений](#211-приведение-к-аккуратной-форме-когда-в-одной-ячейке-хранится-два-и-более-значений)
12. [Приведение к аккуратной форме, когда переменные хранятся в именах столбцов и в значениях](#212-приведение-к-аккуратной-форме-когда-переменные-хранятся-в-именах-столбцов-и-в-значениях)
13. [Приведение к аккуратной форме, когда в одной таблице хранятся несколько типов наблюдений](#213-приведение-к-аккуратной-форме-когда-в-одной-таблице-хранятся-несколько-типов-наблюдений)

## 2.1 Введение

Многие реальные наборы данных требуют существенных преобразований перед тем, как переходить к более детальному анализу. В некоторых случаях весь проект может сводиться к форматированию данных таким образом, чтобы их можно было легко обработать.

Существует множество терминов, описывающих процесс структуризации данных; среди специалистов по данным чаще всего используется термин *tidy data*. Его предложил Хэдли Уикем (Hadley Wickham) для описания такой формы представления данных, которая облегчает анализ. В этой лекции будут рассмотрены многие идеи, сформулированные Хэдли, и способы их реализации в `pandas`.

Что такое *tidy data*? Хэдли формулирует три простых руководящих принципа, по которым определяется, является ли набор данных «аккуратным» («упорядоченным»):
- Каждая переменная образует отдельный столбец
- Каждое наблюдение образует отдельную строку
- Каждый тип наблюдений образует отдельную таблицу

Любой набор данных, не удовлетворяющий этим правилам, считается *«messy»* (неаккуратным). Определение станет более ясным, когда мы начнём перестраивать наши данные в tidy-формат, но сейчас нам нужно понимать, что такое переменные, наблюдения и типы наблюдений.

Чтобы интуитивно понять, что такое переменная, полезно различать ***имя переменной*** и ***её значение***. Имена переменных &ndash; это метки (*label*), такие как *gender*, *race*, *salary*, *position*. Значения переменных &ndash; это то, что может изменяться для каждого наблюдения: например, *male/female* для *gender* или *white/black* для *race*. Одно наблюдение &ndash; это совокупность всех значений переменных для одного типа наблюдений. Чтобы понять, что такое тип наблюдений, можно рассмотреть розничный магазин: у него есть данные по каждой транзакции, сотруднику, клиенту, товару и самому магазину. Каждое из этих понятий можно рассматривать как один тип наблюдений, и для каждого потребуется отдельная таблица. Объединение информации о сотрудниках (например, количество отработанных часов) с информацией о клиентах (например, сумма покупки) в одной таблице нарушает этот принцип `tidy`.

Первый шаг к исправлению неаккуратных данных &ndash; научиться распознавать их, и вариантов тут очень много. Хэдли явно выделяет пять наиболее распространённых типов:
- Имена столбцов &ndash; это значения, а не имена переменных
- Несколько переменных сохранены в именах столбцов
- Переменные сохранены и в строках, и в столбцах
- Несколько типов наблюдений сохранены в одной таблице
- Один тип наблюдений сохранён в нескольких таблицах

Важно понимать, что приведение к аккуратной форме обычно не включает изменение значений, заполнение пропусков или какой-либо анализ. Это именно *изменение формы* или *структуры данных* для соответствия принципам `tidy`. Как только данные приведены в правильную форму, дальнейший анализ упрощается.

Обнаружив неаккуратные данные, можно использовать инструменты `pandas` для их структурирования. Основные инструменты &ndash; это методы `DataFrame`: `stack`, `melt`, `unstack` и `pivot`. Более сложные преобразования требуют разбора текста, для чего используется `str`. Вспомогательные методы, такие как `rename`, `rename_axis`, `reset_index` и `set_index`, помогут довести итоговую структуру до нужного вида.

### Наиболее распространённые случаи

- Преобразование значений переменных, выступающих именами столбцов, с помощью `stack`  
- Преобразование значений переменных, выступающих именами столбцов, с помощью `melt`  
- Одновременное использование `stack` для нескольких групп переменных  
- Инвертирование данных, полученных `stack`/`melt`  
- Использование `unstack` после агрегации с помощью `groupby`  
- Повторение функционала `pivot_table` с помощью агрегации `groupby`  
- Переименование уровней осей для удобного изменения формы данных  
- Приведение к аккуратной форме, когда несколько переменных хранятся в именах столбцов  
- Приведение к аккуратной форме, когда несколько переменных хранятся в значениях столбцов  
- Приведение к аккуратной форме, когда в одной ячейке хранится два и более значений  
- Приведение к аккуратной форме, когда переменные хранятся и в именах столбцов, и в значениях  
- Приведение к аккуратной форме, когда в одной таблице хранятся несколько наблюдательных единиц  

Источники:
- [Wickham, H. Tidy data / H. Wickham // Journal of Statistical Software. - 2014. - V. 59(10). - P. 1–23.](http://vita.had.co.nz/papers/tidy-data.pdf)
- [Tidy data](https://tidyr.tidyverse.org/articles/tidy-data.html)
- [Data tidying: Подготовка наборов данных для анализа на конкретных примерах @ Хабр](https://habr.com/ru/articles/248741/)

## 2.2 Преобразование значений переменных, выступающих именами столбцов, с помощью `stack`

Чтобы лучше понять различия между аккуратными и неаккуратными данными, рассмотрим простую таблицу, которую можно представить как в аккуратной форме, так и нет.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
state_fruit = pd.read_csv('data/state_fruit.csv', index_col=0)
state_fruit

На первый взгляд ничего «неаккуратного» в таблице нет, и информацию легко воспринимать. Однако согласно принципам `tidy` она неаккуратна: каждое имя столбца &ndash; это значение переменной. Более того, имён переменных в `DataFrame` вообще нет. Один из первых шагов &ndash; определить все переменные. В данном наборе это *state* и *fruit*. Численные данные нигде явно не именованы; их можно обозначить как *weight* или любым другим подходящим названием.

### Постановка задачи

В этом наборе значения переменных выступают именами столбцов. Нам нужно преобразовать имена столбцов в значения одного столбца. Используем метод `stack`, чтобы привести `DataFrame` к аккуратной форме.

### Ключевые этапы

1) Названия штатов (*административно-территориальных единиц*) находятся в индексе `DataFrame`. Они уже расположены вертикально и не требуют переработки. Проблема &ndash; в именах столбцов. `stack` берёт все имена столбцов и преобразует их в вертикальный вид как один уровень индекса.

In [ ]:
state_fruit.stack()

In [ ]:
state_fruit.stack().index

2) В результате получаем `Series` с многоуровневым индексом (`MultiIndex`). Исходный индекс сдвинут влево, чтобы освободить место для бывших имён столбцов. Одной этой командой мы фактически получили аккуратные данные: каждая переменная &ndash; *state*, *fruit* и *weight* &ndash; представлена вертикально. Применим `reset_index`, чтобы превратить результат в `DataFrame`.

In [ ]:
state_fruit_tidy = state_fruit.stack().reset_index()
state_fruit_tidy

3) Структура правильная, но имена столбцов неинформативны. Заменим их осмысленными идентификаторами.

In [ ]:
state_fruit_tidy.columns = ['state', 'fruit', 'weight']
state_fruit_tidy

4) Вместо прямого изменения `columns` можно использовать `rename_axis`, чтобы задать имена уровней индекса до `reset_index`.

In [ ]:
state_fruit.stack().rename_axis(['state', 'fruit'])

5) Можно объединить `reset_index` с параметром `name`, чтобы воспроизвести результат шага 3.

In [ ]:
state_fruit.stack().rename_axis(['state', 'fruit']).reset_index(name='weight')

### Справочная информация

`stack` переносит все имена столбцов во внутренний уровень индекса. Каждое старое имя столбца остаётся связанным со «своими» значениями, будучи спаренным с каждым штатом. В исходном `DataFrame` 3×3 было девять значений &ndash; они стали одним `Series` с тем же количеством элементов. Первая строка исходных данных превратилась в первые три значения `Series`.

После `reset_index` (шаг 2) `pandas` по умолчанию называет столбцы `level_0`, `level_1` и `0`, так как у исходного `Series` два безымянных уровня индекса. Уровни индекса нумеруются снаружи внутрь, начиная с нуля.

Шаг 3 показывает простой способ переименования столбцов: присвоить атрибуту `columns` список имён. Альтернативно можно за один шаг задать имена уровней индекса, «сцепив» `rename_axis` (передаём список имён уровней); при `reset_index` `pandas` использует эти имена как новые имена столбцов. Параметр `name` у `reset_index` задаёт имя столбца для значений `Series`.

У всех `Series` есть атрибут `name`, который можно задать напрямую или через `rename`. Именно он станет именем столбца при использовании `reset_index`.

### Дополнительно

Для корректного применения `stack` необходимо поместить все столбцы, которые **НЕ** нужно преобразовывать, в индекс. Если названия штатов не в индексе, вызов `stack` соберёт и их тоже, превратив всё в один длинный `Series`. Правильный порядок: сначала `set_index` для столбцов, которые не следует преобразовывать, затем `stack`.

In [ ]:
state_fruit2 = pd.read_csv('data/state_fruit2.csv')
state_fruit2

In [ ]:
state_fruit2.stack()

In [ ]:
state_fruit2.set_index('Город').stack()

## 2.3 Преобразование значений переменных, выступающих именами столбцов, с помощью `melt`

Как и во многих крупных библиотеках Python, в `pandas` есть несколько способов выполнить одну и ту же задачу, отличающихся читаемостью и производительностью. Метод `DataFrame` `melt` работает подобно `stack`, но предоставляет больше гибкости.

### Постановка задачи

Использовать `melt`, чтобы привести простой DataFrame к аккуратной форме, когда значения переменных находятся в именах столбцов.

### Ключевые этапы

1) Считаем набор `state_fruit2` и определим, какие столбцы нужно преобразовывать, а какие &ndash; нет.  

In [ ]:
state_fruit2 = pd.read_csv('data/state_fruit2.csv')
state_fruit2

2) Вызовем `melt`, передав соответствующие столбцы параметрам `id_vars` и `value_vars`.

In [ ]:
state_fruit2.melt(id_vars=['Город'],
                 value_vars=['Яблоки', 'Апельсины', 'Бананы'])

3) За один шаг сразу получаем аккуратные данные. По умолчанию `melt` называет бывшие имена столбцов `variable`, а соответствующие значения &ndash; `value`. Параметры `var_name` и `value_name` позволяют переименовать эти столбцы.

In [ ]:
state_fruit2.melt(id_vars=['Город'],
                 value_vars=['Яблоки', 'Апельсины', 'Бананы'],
                 var_name='Фрукты',
                 value_name='Вес')

### Справочная информация

`melt` радикально меняет форму `DataFrame`. Ключевые параметры:
- `id_vars` &ndash; список столбцов, которые нужно сохранить вертикальными;  
- `value_vars` &ndash; список столбцов, которые нужно объединить в единственный столбец.

Значения `id_vars` повторяются для каждого столбца из `value_vars`. Важно: `melt` игнорирует индекс &ndash; он тихо сбрасывается и заменяется на `RangeIndex`. Если в индексе есть значимые значения, сначала необходимо выполнить `reset_index`.

### Дополнительно

Все параметры `melt` необязательны: если хотите поместить **все** значения в один столбец, а их прежние имена &ndash; в другой, достаточно вызова по умолчанию. Часто удобнее задавать только `id_vars`, оставляя `value_vars` неуказанным: тогда «объединяются» все остальные столбцы. Если объединяется один столбец, можно передать его имя строкой, без списка.

In [ ]:
state_fruit2.melt()

In [ ]:
state_fruit2.melt(id_vars='Город')

## 2.4 Одновременное использование `stack` для нескольких групп переменных

Некоторые наборы содержат несколько групп переменных в именах столбцов, которые нужно одновременно преобразовать в свои столбцы. Рассмотрим пример с набором фильмов: выберем столбцы с именами актёров и соответствующим числом лайков в социальной сети.

In [ ]:
movie = pd.read_csv('data/movie.csv')
actor = movie[['movie_title', 'actor_1_name', 'actor_2_name', 'actor_3_name', 
               'actor_1_social_network_likes', 'actor_2_social_network_likes', 'actor_3_social_network_likes']]
actor.head()

Если определить переменные как название фильма, имя актёра и число лайков в социальной сети, то нужно независимо преобразовать два набора столбцов; одного вызова `stack` или `melt` недостаточно.


### Постановка задачи

Привести таблицу актёров к аккуратной форме, одновременно применив `stack` к именам актёров и числу лайков в социальной сети, с помощью функции `wide_to_long`.

### Ключевые этапы

1) Используем `wide_to_long` для преобразования. Для этого имена столбцов, которые будем преобразовывать, должны оканчиваться цифрой. Сначала определим функцию, меняющую имена столбцов.

In [ ]:
def change_col_name(col_name):
    col_name = col_name.replace('_name', '')
    if 'social' in col_name:
        fb_idx = col_name.find('social_network_likes')
        col_name = col_name[:5] + col_name[fb_idx - 1:] + col_name[5:fb_idx - 1]
    return col_name

2) Передадим эту функцию в `rename`, чтобы преобразовать имена столбцов.  

In [ ]:
actor2 = actor.rename(columns=change_col_name)
actor2.head()

In [ ]:
actor2.info()

3) Вызовем `wide_to_long`, чтобы одновременно преобразовать группы `actor` и `actor_social_network_likes`.

In [ ]:
stubs = ['actor', 'actor_social_network_likes']
actor2_tidy = pd.wide_to_long(actor2, 
                              stubnames=stubs, 
                              i=['movie_title'], 
                              j='actor_num', 
                              sep='_').reset_index()
actor2_tidy.head()

In [ ]:
actor2_tidy.info()

### Справочная информация

Основной параметр `wide_to_long` &ndash; `stubnames` (список строк). Каждая строка обозначает группу столбцов; все столбцы, начинающиеся с этой строки, преобразуются в один столбец. В нашем примере две группы: `actor` и `actor_social_network_likes`. По умолчанию столбцы в группах должны оканчиваться цифрой &ndash; она становится меткой реструктурированных данных. Если между «заготовкой» и цифрой есть `_`, укажите `sep='_'`.

Имена столбцов изначально не соответствуют шаблону; мы приводим их функцией (например, удаляем суффикс `_name` у актёров и перестраиваем имена с лайками в социальной сети так, чтобы они заканчивались цифрами). `rename` умеет принимать функцию: каждый столбец передаётся в неё по одному.

Кроме того, `wide_to_long` требует уникальную идентификационную переменную `i` (останется вертикальной) и параметр `j` &ndash; имя для числовой метки, снятой с конца исходных имён столбцов. По умолчанию `suffix` &ndash; регулярное выражение `\d+` (одна или более цифр).

### Дополнительно

`wide_to_long` работает, когда все группы имеют одинаковые числовые окончания. Если окончания различаются или это не цифры, всё равно можно применить `wide_to_long`: переименуйте столбцы так, чтобы они оканчивались нужными метками, и измените `suffix`, например на `.*`, чтобы матчить произвольный хвост.

In [ ]:
df = pd.read_csv('data/stackme.csv')
df

In [ ]:
df2 = df.rename(columns = {'a1':'group1_a1', 'b2':'group1_b2',
                           'd':'group2_a1', 'e':'group2_b2'})
df2

In [ ]:
pd.wide_to_long(df2, 
                stubnames=['group1', 'group2'], 
                i=['State', 'Country', 'Test'], 
                j='Label', 
                suffix='.+', 
                sep='_')

## 2.5 Инвертирование данных, полученных `stack`/`melt`

У DataFrame есть пары методов: `stack`/`unstack` и `melt`/`pivot`, которые преобразуют горизонтальные имена столбцов в вертикальные значения и обратно. Пара `stack`/`unstack` управляет индексами строк/столбцов; `melt`/`pivot` даёт более гибкий выбор столбцов.

### Постановка задачи

Выполнить `stack`/`melt` над набором данных и сразу инвертировать операцию с помощью `unstack`/`pivot`, вернув исходную форму.

### Ключевые этапы

1) Считаем набор `college` с названием учреждения в индексе, оставив только столбцы по расовой структуре бакалавриата. 

In [ ]:
usecol_func = lambda x: 'UGDS_' in x or x == 'INSTNM'
college = pd.read_csv('data/college.csv', 
                      index_col='INSTNM', 
                      usecols=usecol_func)
college.head()

2) Применим `stack`, чтобы превратить каждое имя столбца во внутренний уровень индекса.  

In [ ]:
college_stacked = college.stack()
college_stacked.head(18)

3) Инвертируем результат методом `unstack` у `Series`.  

In [ ]:
college_stacked.unstack().head()

4) Повторим логику с `melt` и `pivot`: прочитаем данные без помещения названия учреждения в индекс.

In [ ]:
college2 = pd.read_csv('data/college.csv', 
                       usecols=usecol_func)
college2.head()

5) Используем `melt`, чтобы объединить все «расовые» столбцы в один.  

In [ ]:
college_melted = college2.melt(id_vars='INSTNM', 
                               var_name='Race',
                               value_name='Percentage')
college_melted.head()

6) Используем `pivot`, чтобы инвертировать результат предыдущего шага. 

In [ ]:
melted_inv = college_melted.pivot(index='INSTNM',
                                  columns='Race',
                                  values='Percentage')
melted_inv.head()

7) Чтобы получить точную копию исходной структуры из шага 4, отсортируем строки и столбцы через `.loc`, затем сделаем `reset_index`.

In [ ]:
college2_replication = melted_inv.loc[college2['INSTNM'], college2.columns[1:]].reset_index()
college2.equals(college2_replication)

### Справочная информация

В шаге 1 демонстрируется гибкость `read_csv`: `usecols` может принимать список столбцов или функцию (на вход &ndash; имя столбца, на выход &ndash; булево). Это помогает экономить память.

`stack` (шаг 2) помещает имена столбцов во внутренний уровень индекса и возвращает `Series`. `unstack` (шаг 3) берёт значения внутреннего уровня индекса и превращает их в имена столбцов.

> Замечание: результат шага 3 не полностью совпадает с шагом 1, так как `stack` по умолчанию отбрасывает целые строки пропусков. Чтобы сохранить их, используйте `dropna=False` в `stack`.

`melt` (шаг 5) объединяет все «расовые» столбцы в один (если `value_vars=None`, объединяются все столбцы, не входящие в `id_vars`). `pivot` (шаг 6) принимает три строковых параметра: `index`, `columns`, `values`. Столбец из `index` становится индексом, значения `columns` &ndash; именами столбцов, значения `values` раскладываются по пересечениям.

Чтобы получить точную копию при `pivot`, отсортируйте строки и столбцы в исходном порядке (см. шаг 7).

### Дополнительно

По умолчанию `unstack` использует внутренний уровень индекса (`level=-1`). Можно развернуть внешний уровень, указав `level=0`. Для чистого транспонирования `DataFrame` не обязательно использовать `stack`/`unstack` &ndash; достаточно `transpose()` или атрибута `T`.

In [ ]:
college.stack().unstack(0)

In [ ]:
college.T

## 2.6 Использование `unstack` после агрегации `groupby`

Группировка по одному столбцу с агрегацией по одному столбцу даёт простой и наглядный результат. При группировке по нескольким столбцам результат может быть менее удобен для восприятия. Поскольку `groupby` по умолчанию помещает уникальные значения группирующих столбцов в индекс, `unstack` помогает переставить данные в более удобный для сравнения вид.

### Постановка задачи

Используя набор `employee`, выполнить агрегацию с группировкой по нескольким столбцам, затем применить `unstack`.

### Ключевые этапы

1) Прочитаем `employee` и найдем среднюю зарплату по расам.

In [ ]:
employee = pd.read_csv('data/employee.csv')
employee.head()

In [ ]:
employee.groupby('RACE')['BASE_SALARY'].mean().astype(int)

2) Теперь найдем среднюю зарплату по расам и полу одновременно. 

In [ ]:
agg = employee.groupby(['RACE', 'GENDER'])['BASE_SALARY'].mean().astype(int)
agg

3) Чтобы легче сравнивать мужские и женские зарплаты внутри каждой расы, примените `unstack` к уровню `gender`.

In [ ]:
agg.unstack('GENDER')

4) Аналогично примените `unstack` к уровню `race`.

In [ ]:
agg.unstack('RACE')

### Справочная информация

- Шаг 1 &ndash; одна группировка (*RACE*), один агрегируемый столбец (*BASE_SALARY*), одна функция (`mean`).
- Шаг 2 &ndash; группировка по двум измерениям (раса и пол) даёт `Series` с `MultiIndex`, что затрудняет сравнение. `unstack` переносит один из уровней индекса в столбцы. По умолчанию используется внутренний уровень; нужный можно указать параметром `level` (по имени уровня предпочтительнее).

### Дополнительно

Если группирующих и агрегируемых столбцов несколько, результатом сразу будет `DataFrame` (`MultiIndex` в строках и/или столбцах). Можно последовательно применять `unstack` и `stack`, пока не получите желаемую структуру.

In [ ]:
agg2 = employee.groupby(['RACE', 'GENDER'])['BASE_SALARY'].agg(['mean', 'max', 'min']).astype(int)
agg2

In [ ]:
agg2.unstack('GENDER')

## 2.7 Повторение функционала `pivot_table` с помощью агрегации `groupby`

На первый взгляд `pivot_table` &ndash; уникальный способ анализа данных. Однако его можно воспроизвести с помощью `groupby` и последующего `unstack`.

### Постановка задачи

Используя набор `flights`, создать сводную таблицу `pivot_table`, затем создать её средствами `groupby`.

### Ключевые этапы

1) Прочитаем `flights` и используем `pivot_table`, чтобы получить общее число отменённых рейсов по аэропорту вылета для каждой авиакомпании (задавая `index`, `columns`, `values`, `aggfunc='sum'`, при необходимости `fill_value=0`).

In [ ]:
flights = pd.read_csv('data/flights.csv')
flights.head()

In [ ]:
fp = flights.pivot_table(index='AIRLINE', 
                         columns='ORG_AIR', 
                         values='CANCELLED', 
                         aggfunc='sum',
                         fill_value=0).round(2)
fp.head()

2) Для воспроизведения начнём с `groupby` по всем столбцам, переданным в `index` и `columns`.  

In [ ]:
fg = flights.groupby(['AIRLINE', 'ORG_AIR'])['CANCELLED'].sum()
fg.head()

In [ ]:
fg.index

3) Применим `unstack`, чтобы перенести уровень `ORG_AIR` из индекса в имена столбцов; при необходимости используем `fill_value=0`.

In [ ]:
fg_unstack = fg.unstack('ORG_AIR', fill_value=0)
fg_unstack.head()

In [ ]:
fp.equals(fg_unstack)

### Справочная информация

`pivot_table` фактически агрегирует по пересечениям уникальных комбинаций столбцов из `index` и `columns`. Воспроизведение: сгруппировать по тем же столбцам, агрегировать `values`, затем `unstack` нужный уровень индекса в столбцы.

### Дополнительно

In [ ]:
fp2 = flights.pivot_table(index=['AIRLINE', 'MONTH'],
                          columns=['ORG_AIR', 'CANCELLED'],
                          values=['DEP_DELAY', 'DIST'],
                          aggfunc=[np.mean, np.sum],
                          fill_value=0)
fp2.head()

In [ ]:
flights.groupby(['AIRLINE', 'MONTH', 'ORG_AIR', 'CANCELLED'])[['DEP_DELAY', 'DIST']] \
       .agg(['mean', 'sum']) \
       .unstack(['ORG_AIR', 'CANCELLED'], fill_value=0) \
       .swaplevel(0, 1, axis='columns') \
       .head()

## 2.8 Переименование уровней осей для удобного изменения формы данных

Преобразования с помощью `stack`/`unstack` проще, когда у каждого уровня осей (индекса/столбцов) есть имя. `Pandas` позволяет ссылаться на уровень по позиции или по имени; лучше использовать имена (явный способ предпочтительнее неявного).

## Постановка задачи

Задать имена уровням и в явном виде управлять структурой данных через `stack`/`unstack`.

### Ключевые этапы

1) Считаем `college` и посчитаем сводные показатели по численности бакалавриата и баллам `SAT` по математике по учреждениям и религиозной принадлежности.

In [ ]:
college = pd.read_csv('data/college.csv')
college

In [ ]:
cg = college.groupby(['STABBR', 'RELAFFIL'])[['UGDS', 'SATMTMID']] \
            .agg(['count', 'min', 'max']).head(6)
cg

2) Уровни индекса имеют имена (бывшие имена столбцов). Для столбцов без имён &ndash; зададим имена с помощью `rename_axis`.

In [ ]:
cg = cg.rename_axis(['AGG_COLS', 'AGG_FUNCS'], axis='columns')
cg

3) Применим `stack`, чтобы переместить уровень столбцов `AGG_FUNCS` в уровень индекса.  

In [ ]:
cg.stack('AGG_FUNCS').head(6)

4) По умолчанию новый уровень становится внутренним &ndash; используем `swaplevel`, чтобы поменять уровни местами.

In [ ]:
cg.stack('AGG_FUNCS').swaplevel('AGG_FUNCS', 'STABBR', axis='index').head(6)

5) Отсортируем уровни через `sort_index`.  

In [ ]:
cg.stack('AGG_FUNCS') \
  .swaplevel('AGG_FUNCS', 'STABBR', axis='index') \
  .sort_index(level='RELAFFIL', axis='index') \
  .sort_index(level='AGG_COLS', axis='columns').head(6)

6) Комбинируем `stack` для одних уровней и `unstack` для других в одной цепочке операций.  

In [ ]:
cg.stack('AGG_FUNCS').unstack(['RELAFFIL', 'STABBR'])

7) Чтобы получить `Series`, можно перенести все уровни столбцов в индекс одновременно.

In [ ]:
cg.stack(['AGG_FUNCS', 'AGG_COLS']).head(12)

### Справочная информация

`rename_axis` способен менять как имена уровней (если передать список/скаляр), так и значения уровней (если передать словарь/функцию). После именования уровней управление формой данных становится явным: `stack` переносит указанный уровень столбцов в индекс, `swaplevel` меняет порядок уровней, `sort_index` сортирует значения уровней.

### Дополнительно

Если хотите избавиться от имён уровней для уменьшения «визуального шума», установите их в `None`.

In [ ]:
cg.rename_axis([None, None], axis='index').rename_axis([None, None], axis='columns')

## 2.9 Приведение к аккуратной форме, когда несколько переменных хранятся в именах столбцов

Один из частых видов неаккуратных данных &ndash; когда имена столбцов содержат несколько переменных (например, пол и возраст, склеенные вместе). Чтобы привести такой набор к аккуратной форме, манипулируем именами столбцов с помощью аксессора `str`.

### Постановка задачи

Идентифицировать переменные (часть из них склеена в именах столбцов), затем преобразовать текст, извлекая корректные значения переменных.

### Ключевые этапы

1) Прочитаем набор по мужской тяжёлой атлетике и определим переменные.

In [ ]:
weightlifting = pd.read_csv('data/weightlifting_men.csv')
weightlifting

2) Переменные: весовая категория, категория «пол/возраст», квалификационный итог. Пол и возраст склеены. Сначала используем `melt`, чтобы перенести имена столбцов «возраст/пол» в один вертикальный столбец.  

In [ ]:
wl_melt = weightlifting.melt(id_vars='Weight Category', 
                             var_name='sex_age', 
                             value_name='Qual Total')
wl_melt.head()

3) Выберем столбец `sex_age` и применим `str.split` для разделения на два столбца.  

In [ ]:
sex_age = wl_melt['sex_age'].str.split(expand=True)
sex_age.head()

4) Переименуем полученные столбцы в осмысленные имена.  

In [ ]:
sex_age.columns = ['Sex', 'Age Group']
sex_age.head()

5) Используем индексацию после `str`, чтобы взять первый символ из столбца `Sex`.  

In [ ]:
sex_age['Sex'] = sex_age['Sex'].str[0]
sex_age.head()

6) Объединим результат с `wl_melt` через `pd.concat` (горизонтально), чтобы получить аккуратный набор.  

In [ ]:
wl_cat_total = wl_melt[['Weight Category', 'Qual Total']]
wl_tidy = pd.concat([sex_age, wl_cat_total], axis='columns')
wl_tidy.head()

7) Альтернативная цепочка даёт тот же результат (вариант записи операций).

In [ ]:
cols = ['Weight Category', 'Qual Total']
sex_age[cols] = wl_melt[cols]
sex_age

### Справочная информация

Когда переменные «зашиты» в именах столбцов, используйте `melt` (или `stack`). Переменная «Весовая категория» уже на месте &ndash; передаём её в `id_vars`. Столбец `sex_age` нужно распарсить. `str.split` по умолчанию делит по пробелу; можно задать строку или регулярное выражение (`pat`). При `expand=True` каждая часть попадает в отдельный столбец. Индексация через `str[...]` позволяет извлекать подстроки (например, первый символ для пола). Объединяем части через `concat`.

### Дополнительно

Можно обойтись без `split`, используя `assign` и `str.extract` с регулярным выражением и группами захвата. Пример: извлечь возрастной интервал вида `\d{2}[+-](?:\d{2})?`. После формирования аккуратных столбцов исходный `sex_age` удаляется. Результаты, полученные разными путями, можно сравнить на эквивалентность.

In [ ]:
age_group = wl_melt.sex_age.str.extract('(\\d{2}[-+](?:\\d{2})?)', expand=False)
sex = wl_melt.sex_age.str[0]
new_cols = {'Sex': sex, 
            'Age Group': age_group}

In [ ]:
wl_tidy2 = wl_melt.assign(**new_cols).drop('sex_age', axis='columns')
wl_tidy2.head()

In [ ]:
wl_tidy2.sort_index(axis=1).equals(wl_tidy.sort_index(axis=1))

## 2.10 Приведение к аккуратной форме, когда несколько переменных хранятся в одном столбце

В аккуратных наборах на каждую переменную приходится отдельный столбец. Иногда несколько имён переменных помещают в один столбец, а соответствующее значение &ndash; в другой. Тогда каждая логическая запись «растянута» по нескольким строкам.

### Постановка задачи

Определить столбец с неправильно структурированными переменными и использовать преобразования, чтобы получить аккуратные данные.

### Ключевые этапы

1) Прочитаем набор по проверкам ресторанов и преобразуем тип столбца `Date` в `datetime64`.

In [ ]:
inspections = pd.read_csv('data/restaurant_inspections.csv', parse_dates=['Date'])
inspections.head(10)

2) Столбцы `Name` и `Date` &ndash; корректные переменные. Столбец `Info` фактически хранит пять разных переменных: `Borough`, `Cuisine`, `Description`, `Grade`, `Score`. Используем `pivot`, чтобы оставить `Name` и `Date` вертикально, создать новые столбцы из значений `Info` и использовать столбец `Value` как значения пересечений.  

In [ ]:
inspections.pivot(index=['Name', 'Date'], columns='Info', values='Value')

3) Альтернативный вариант: перенесём `Name`, `Date` и `Info` в индекс через `set_index`.  

In [ ]:
inspections.set_index(['Name','Date', 'Info']).head(10)

4) Применим `unstack`, чтобы «разложить» значения `Info` по столбцам.  

In [ ]:
inspections.set_index(['Name','Date', 'Info']).unstack('Info').head()

5) Вернём уровни индекса в столбцы через `reset_index`.  

In [ ]:
insp_tidy = inspections.set_index(['Name','Date', 'Info']) \
                       .unstack('Info') \
                       .reset_index(col_level=-1)
insp_tidy.head()

6) Уберём «служебные» уровни: применим `droplevel` к столбцовому `MultiIndex` и переименуем имя уровня в `None`.  

In [ ]:
insp_tidy.columns = insp_tidy.columns.droplevel(0).rename(None)
insp_tidy.head()

7) Этого `MultiIndex` можно было избежать, если преобразовать одно-столбцовый `DataFrame` в `Series` методом `squeeze` перед `unstack`.

In [ ]:
inspections.set_index(['Name','Date', 'Info']) \
           .squeeze() \
           .unstack('Info') \
           .reset_index() \
           .rename_axis(None, axis='columns')

### Справочная информация

В качестве альтернативы `pivot` можно использовать `unstack`, который работает с уровнями индекса. Сначала переносим и «перекладываемые», и «неперекладываемые» столбцы в индекс (`set_index`), затем применяем `unstack`. При `unstack` `DataFrame` `pandas` сохраняет исходное имя столбца (здесь это `Value`) и создаёт столбцовый `MultiIndex`. Далее `reset_index` возвращает «внешние» уровни индекса в столбцы; параметр `col_level=-1` позволяет разместить имена на нижнем уровне. Очищаем оставшийся `MultiIndex` через `droplevel` и снимаем имя уровня (`None`).

## 2.11 Приведение к аккуратной форме, когда в одной ячейке хранится два и более значений

Табличные данные двумерны, поэтому объём информации в одной ячейке ограничен. Иногда встречаются наборы, где в ячейке хранится несколько значений. Аккуратные данные допускают ровно одно значение в каждой ячейке. Чтобы исправить ситуацию, обычно нужно распарсить строку в несколько столбцов методами `str`.

### Постановка задачи

Преобразовать набор, где один столбец содержит в каждой ячейке несколько переменных. Разделить строки на отдельные столбцы, чтобы получить аккуратную форму.

### Ключевые этапы

1) Прочитаем набор и определим переменные.

In [ ]:
cities = pd.read_csv('data/texas_cities.csv')
cities

2) `City` корректен (одна величина). Столбец `Geolocation` содержит четыре переменные: широта, направление широты, долгота, направление долготы. Разделим `Geolocation` на четыре столбца с помощью `str.split` по подходящему шаблону.  

In [ ]:
geolocations = cities.Geolocation.str.split(pat='. ', expand=True)
geolocations.columns = ['latitude', 'latitude direction', 'longitude', 'longitude direction']
geolocations

3) Поскольку исходный тип `Geolocation` &ndash; `str`, новые столбцы тоже будут `str`. Преобразуем широту и долготу в `float`.

In [ ]:
geolocations.info()

In [ ]:
geolocations = geolocations.astype({'latitude': 'float', 'longitude': 'float'})
geolocations.dtypes

4) Объедините новые столбцы со столбцом `City`.

In [ ]:
cities_tidy = pd.concat([cities['City'], geolocations], axis='columns')
cities_tidy

### Справочная информация

Мы выбрали разбиение на четыре столбца, но можно было оставить два числовых столбца (широта и долгота), используя знак для направлений. Простой путь &ndash; `str.split` с регулярным выражением, задающим позиции разбиения (например, «символ градуса + пробел», затем «запятая + пробел»). В итоге три разбиения &ndash; четыре столбца. Затем конвертируем типы (`to_numeric`/`astype`). Объединяем с исходным столбцом города.

## 2.12 Приведение к аккуратной форме, когда переменные хранятся в именах столбцов и в значениях

Сложный вариант неаккуратных данных &ndash; когда часть переменных хранится горизонтально (в именах столбцов), а часть &ndash; вертикально (в значениях). Часто это результат уже подготовленного сводного отчёта.

### Постановка задачи

Идентифицировать переменные, заданные вертикально и горизонтально, и привести данные к аккуратной форме методами `melt` и `pivot_table`.

### Ключевые этапы

1) Прочитаем набор `sensors` и определим переменные.

In [ ]:
sensors = pd.read_csv('data/sensors.csv')
sensors

2) Корректно вертикально расположен только `Group`. Столбец `Property` имеет три уникальные переменные: `Pressure`, `Temperature`, `Flow`. Остальные столбцы (`2012`–`2016`) &ndash; это одна переменная «Год» (`Year`). Одним методом `DataFrame` такую структуру не перестроить. Сначала используем `melt`, чтобы перенести годы в отдельный столбец.

In [ ]:
sensors.melt(id_vars=['Group', 'Property'], var_name='Year').head(6)

3) Затем используем `pivot_table`, чтобы значения `Property` стали именами столбцов.

In [ ]:
sensors.melt(id_vars=['Group', 'Property'], var_name='Year') \
       .pivot_table(index=['Group', 'Year'], columns='Property', values='value') \
       .reset_index() \
       .rename_axis(None, axis='columns')

### Справочная информация

`Pandas` не умеет одновременно «поворачивать» несколько наборов столбцов, поэтому действуем по шагам. Сначала исправляем годы (`melt`, оставляя `Property` в `id_vars`). Полученный результат соответствует шаблону, где несколько переменных хранятся в значениях &ndash; для «поворота» используем `pivot_table` (можно указать несколько столбцов в `index`). После «поворота» переменные `Group` и `Year` оказываются в индексе &ndash; вернём их в столбцы (`reset_index`). Имя уровня столбцов, унаследованное от `columns`, после `reset_index` становится лишним &ndash; удалим его (`rename_axis(None, axis='columns')`).

### Дополнительно

Почти всегда есть альтернативный путь через `stack`/`unstack`: сначала перенесите столбцы, которые не «поворачиваются» сейчас, в индекс, затем применяйте нужную пару методов.

In [ ]:
sensors.set_index(['Group', 'Property']) \
       .stack() \
       .unstack('Property') \
       .rename_axis(['Group', 'Year'], axis='index') \
       .rename_axis(None, axis='columns') \
       .reset_index()

## 2.13 Приведение к аккуратной форме, когда в одной таблице хранятся несколько типов наблюдений

Данные проще сопровождать, когда каждая таблица содержит информацию только об одной наблюдательной единице. С другой стороны, для анализа иногда удобнее иметь всё в одной таблице, а для машинного обучения &ndash; тем более. Цель подхода tidy &ndash; не непосредственный анализ, а такая структура, которая упрощает последующий анализ. Если в одной таблице присутствуют несколько типов наблюдений, их может понадобиться разделить на отдельные таблицы.

### Постановка задачи

В наборе `movie` выделить три типа наблюдений (*фильмы*, *актёры*, *режиссёры*) и создать отдельные таблицы. Важно понимать, что число лайков актёров и режиссёров в социальной сети **независимо** от фильма: каждому актёру/режиссёру сопоставлено одно значение лайков. Благодаря этой независимости данные можно разделить. В реляционных БД такой процесс называется нормализацией: он повышает целостность данных и снижает избыточность.

### Ключевые этапы

1) Прочитаем модифицированный набор `movie` и вывести первые строки.  

In [ ]:
movie = pd.read_csv('data/movie_altered.csv')
movie.head()

2) Набор содержит сведения о фильме, режиссёре и актёрах &ndash; это три типа наблюдений. Сначала создадим уникальный идентификатор фильма (`insert` столбца `id`).  

In [ ]:
movie.insert(0, 'id', np.arange(len(movie)))
movie.head()

3) Используем `wide_to_long`, чтобы собрать всех актёров в один столбец, их лайки в социальной сети &ndash; в другой; аналогично для режиссёра (хотя он один на фильм).  

In [ ]:
stubnames = ['director', 'director_social_likes', 'actor', 'actor_social_likes']
movie_long = pd.wide_to_long(movie, 
                             stubnames=stubnames, 
                             i='id', 
                             j='num', 
                             sep='_').reset_index()
movie_long['num'] = movie_long['num'].astype(int)
movie_long.head(9)

4) Теперь данные готовы к разбиению на несколько меньших таблиц (отдельно *фильмы*, *актёры*, *режиссёры*), сохраняя `id` и служебный номер `num` исходной позиции.  

In [ ]:
movie_table = movie_long[['id','title', 'year', 'duration', 'rating']]
director_table = movie_long[['id', 'director', 'num', 'director_social_likes']]
actor_table = movie_long[['id', 'actor', 'num', 'actor_social_likes']]

In [ ]:
movie_table.head(9)

In [ ]:
director_table.head(9)

In [ ]:
actor_table.head(9)